# Cobre Quickstart — End-to-End Python Tutorial

This notebook walks through the complete Python workflow for the **Cobre** power
system solver:

1. Load and validate an input case
2. Run the SDDP solver
3. Analyse convergence with **polars** and **matplotlib**
4. Explore simulation cost results

The notebook uses the `1dtoy` example case bundled with the repository —
a small single-bus, single-hydro system that solves in a few seconds.

## 1. Installation

Install the required packages if you haven't already:

```bash
pip install cobre polars matplotlib pyarrow
```

The cell below verifies that `cobre` is importable and prints its version.

In [ ]:
try:
    import cobre
except ImportError as exc:
    raise ImportError(
        "The 'cobre' package is not installed.\n"
        "Install it with: pip install cobre"
    ) from exc

import polars as pl
import matplotlib.pyplot as plt

print(f"Cobre version: {cobre.__version__}")
print(f"Polars version: {pl.__version__}")

## 2. Loading and Validating a Case

`cobre.io.load_case` reads all input files for a case directory and returns a
system object describing buses, generators, hydros, and network topology.

`cobre.io.validate` runs the five-layer validation pipeline and raises if the
case is malformed.

In [ ]:
CASE_DIR = "../1dtoy/"

system = cobre.io.load_case(CASE_DIR)
print(f"Buses    : {len(system.buses)}")
print(f"Thermals : {len(system.thermals)}")
print(f"Hydros   : {len(system.hydros)}")
print(f"Lines    : {len(system.lines)}")

In [ ]:
cobre.io.validate(CASE_DIR)
print("Case validation passed.")

## 3. Running the Solver

`cobre.run.run` executes the full SDDP training and simulation pipeline.
It returns a dict with convergence metadata and the path to the output directory.

| Key | Meaning |
|---|---|
| `converged` | Whether a stopping rule was triggered |
| `iterations` | Number of training iterations |
| `lower_bound` | Final lower bound (expected cost) |
| `gap_percent` | Relative optimality gap in % |
| `output_dir` | Path where Parquet results were written |

The `1dtoy` case typically converges in under 30 seconds on a laptop.

In [ ]:
result = cobre.run.run(CASE_DIR, threads=2)

print(f"Converged  : {result['converged']}")
print(f"Iterations : {result['iterations']}")
print(f"Lower bound: {result['lower_bound']:.2f}")
print(f"Gap        : {result['gap_percent']:.4f}%")
print(f"Output dir : {result['output_dir']}")

## 4. Convergence Analysis

`cobre.results.load_convergence_arrow` returns a **PyArrow Table** with one row
per training iteration. We convert it to a **polars DataFrame** for ergonomic
filtering and aggregation.

Typical columns: `iteration`, `lower_bound`, `upper_bound_mean`,
`upper_bound_std`, `gap_percent`.

In [ ]:
conv_arrow = cobre.results.load_convergence_arrow(result["output_dir"])
df_conv = pl.from_arrow(conv_arrow)
print(f"Convergence table: {df_conv.shape[0]} iterations x {df_conv.shape[1]} columns")
df_conv.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    df_conv["iteration"].to_list(),
    df_conv["lower_bound"].to_list(),
    label="Lower Bound",
    linewidth=2,
)
ax.plot(
    df_conv["iteration"].to_list(),
    df_conv["upper_bound_mean"].to_list(),
    label="Upper Bound (mean)",
    linestyle="--",
    linewidth=2,
)

ax.set_xlabel("Iteration")
ax.set_ylabel("Expected Cost ($)")
ax.set_title("SDDP Convergence — 1dtoy")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Simulation Results

After training, the solver runs a Monte Carlo simulation to evaluate the policy.
`cobre.results.load_simulation_arrow` loads the simulation output for a given
entity type. Here we load `"costs"` — the per-stage, per-scenario operating
costs.

Typical columns: `scenario_id`, `stage_id`, `total_cost`.

In [ ]:
costs_arrow = cobre.results.load_simulation_arrow(
    result["output_dir"], entity_type="costs"
)
df_costs = pl.from_arrow(costs_arrow)
print(f"Simulation costs: {df_costs.shape[0]} rows x {df_costs.shape[1]} columns")
df_costs.head()

In [ ]:
stage_costs = (
    df_costs.group_by("stage_id")
    .agg(
        pl.col("total_cost").mean().alias("mean_cost"),
        pl.col("total_cost").std().alias("std_cost"),
    )
    .sort("stage_id")
)
print("Mean operating cost per stage:")
stage_costs

In [ ]:
# Compute total cost per scenario (sum across all stages)
total_per_scenario = (
    df_costs.group_by("scenario_id")
    .agg(pl.col("total_cost").sum().alias("total_cost"))
    .sort("scenario_id")
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
    total_per_scenario["total_cost"].to_list(),
    bins=20,
    edgecolor="white",
    linewidth=0.6,
)
ax.set_xlabel("Total Operating Cost ($)")
ax.set_ylabel("Number of Scenarios")
ax.set_title("Distribution of Total Operating Cost — 1dtoy")
plt.tight_layout()
plt.show()

## 6. Cleanup and Next Steps

The solver writes output files to `../1dtoy/output/` by default. Remove them
if you want a clean slate for the next run.

In [ ]:
import shutil
from pathlib import Path

output_path = Path(result["output_dir"])
if output_path.exists():
    shutil.rmtree(output_path)
    print(f"Removed output directory: {output_path}")
else:
    print("Output directory already removed.")

### What to Explore Next

- **Larger cases**: try the `4ree/` example for a multi-reservoir system
- **Parallel execution**: increase `threads=` or run with MPI (`mpirun -n 4 cobre run ...`)
- **Other entity types**: pass `entity_type="hydros"`, `"thermals"`, or `"buses"` to
  `load_simulation_arrow` for element-level results
- **Software book**: full documentation at <https://cobre-rs.github.io/cobre/>